# 12 — Decorators, Context Managers, Iterators, and Generators

Goal: master Python’s compositional building blocks for reusable behavior.

_Generated: 2026-02-19_

## Setup

This course targets **Python 3.11+** (works on 3.10+, with a few feature differences).

Recommended tooling:

```bash
# create + activate a virtual environment
python -m venv .venv
# mac/linux:
source .venv/bin/activate
# windows (PowerShell):
# .venv\Scripts\Activate.ps1

python -m pip install -U pip

# quality-of-life (optional but recommended)
python -m pip install -U ipykernel ruff black pytest mypy
```

If you're using Jupyter:
```bash
python -m ipykernel install --user --name python-course --display-name "Python Course (.venv)"
```

In [ ]:

import sys, platform, os
print("python:", sys.version.split()[0])
print("implementation:", platform.python_implementation())
print("platform:", platform.platform())
print("cwd:", os.getcwd())


## 1.
L1: Decorators (wrapping functions)

A decorator is a callable that takes a function and returns a function.

Critical: preserve metadata with `functools.wraps`.

In [ ]:

from functools import wraps
import time

def timing(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        out = fn(*args, **kwargs)
        dt = time.perf_counter() - t0
        print(f"{fn.__name__} took {dt*1000:.2f} ms")
        return out
    return wrapper

@timing
def slow_add(n: int) -> int:
    s = 0
    for i in range(n):
        s += i
    return s

print(slow_add(100_000))
print("name preserved:", slow_add.__name__)


## 2.
L2: Parameterized decorators

A parameterized decorator is “decorator factory”.

In [ ]:

from functools import wraps

def repeat(times: int):
    def dec(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            out = None
            for _ in range(times):
                out = fn(*args, **kwargs)
            return out
        return wrapper
    return dec

@repeat(3)
def ping():
    print("ping")
    return "done"

print(ping())


## 3.
L3: Context managers (`with`) in depth

Context managers guarantee cleanup.

Two common ways to create them:
- implement `__enter__/__exit__`
- use `contextlib.contextmanager`

In [ ]:

from contextlib import contextmanager

@contextmanager
def managed_resource(name: str):
    print("acquire", name)
    try:
        yield {"name": name}
    finally:
        print("release", name)

with managed_resource("db-conn") as r:
    print("using", r)


## 4.
L4: Iterators and iterables

- An **iterable** can produce an iterator (`iter(x)` works).
- An **iterator** produces values via `__next__` until `StopIteration`.

This is the foundation of `for`.

In [ ]:

class CountTo:
    def __init__(self, n: int):
        self.n = n
        self.i = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.i >= self.n:
            raise StopIteration
        self.i += 1
        return self.i

print(list(CountTo(5)))


## 5.
L5: Generators (simpler iterators)

A generator function uses `yield`.
Generators are ideal for streaming pipelines and large data.

In [ ]:

def gen_squares(n: int):
    for i in range(n):
        yield i * i

print(list(gen_squares(5)))


## 6.
L6: `yield from` and generator composition

`yield from` delegates iteration to another iterator/generator.

In [ ]:

def chain(*iters):
    for it in iters:
        yield from it

print(list(chain([1,2], (3,4), "ab")))


## 7.
L7: Exercises

1. Write a decorator `require_positive` that raises `ValueError` if any numeric arg is <= 0.
2. Write a generator `read_lines(path)` that yields lines stripped of newline.
3. Write a context manager that times a block (like a “with timer(): ...”).

## 8.
L8: Generator advanced: `send`, `throw`, and `close`

Generators can receive values via `.send()` and can be closed.
Most code uses only `yield`, but knowing this helps when reading advanced libs.

In [ ]:

def echo():
    x = yield "ready"
    while True:
        x = yield x

g = echo()
print(next(g))       # prime -> "ready"
print(g.send(10))    # yields 10
print(g.send("hi"))  # yields "hi"
g.close()


## 9.
L9: `contextlib.ExitStack` for dynamic resource management

Useful when you create resources in a loop and want guaranteed cleanup.

In [ ]:

from contextlib import ExitStack
from pathlib import Path

with ExitStack() as stack:
    fs = [stack.enter_context(Path(f"s{i}.txt").open("w", encoding="utf-8")) for i in range(2)]
    for i, f in enumerate(fs):
        f.write(f"{i}\n")

for i in range(2):
    Path(f"s{i}.txt").unlink(missing_ok=True)
print("ok")
